# 03 · Hipóteses e diretrizes para a modelagem

Este notebook transforma os achados exploratórios em hipóteses testáveis. Cada bloco separa quatro elementos:

1. proposição;
2. evidência observada;
3. limitação da interpretação;
4. diretriz para o pipeline.

A análise evita tratar associação como causalidade e verifica disponibilidade temporal, redundância, estabilidade entre anos e risco de leakage.


In [1]:
from pathlib import Path
import os
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

SEED = 42

def localizar_raiz(inicio: Path | None = None) -> Path:
    atual = (inicio or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "data" / "processed").exists():
            return candidato
    raise FileNotFoundError("Não foi encontrada a pasta data/processed.")

PROJECT_ROOT = localizar_raiz()

# Garante que o pacote src seja localizado mesmo quando o notebook estiver
# dentro de notebooks/analysis ou for executado pelo Jupyter/NBConvert.
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Padroniza caminhos relativos usados por outros módulos do projeto.
os.chdir(PROJECT_ROOT)

PROCESSED = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"
IMAGES.mkdir(parents=True, exist_ok=True)

from src.visualization.plot_style import (
    aplicar_estilo, salvar_figura, formatar_percentual,
    AZUL, LARANJA, CINZA, VERDE, VERMELHO, CORES_REGIAO,
)
from src.visualization.business_labels import rotulo, rotular

aplicar_estilo()
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

ARQ_MUN = PROCESSED / "base_analitica_municipio.parquet"
ARQ_ALU = PROCESSED / "base_analitica_aluno.parquet"
if not ARQ_MUN.exists() or not ARQ_ALU.exists():
    raise FileNotFoundError("Execute primeiro a criação das bases analíticas.")

abt_mun = pd.read_parquet(ARQ_MUN)
mun = abt_mun[abt_mun["atingiu_meta_t"].notna()].copy()
mun["atingiu"] = mun["atingiu_meta_t"].astype(int)
mun["ano_alvo"] = pd.to_numeric(mun["ano_alvo"], errors="coerce").astype("Int64")
print("Base municipal observada:", mun.shape)
print(mun.groupby("ano_alvo", observed=True).size().to_dict())


Base municipal observada: (10786, 60)
{np.int64(2024): 5347, np.int64(2025): 5439}


## 1. Funções para testes e registro de decisões


In [2]:
resultados=[]

def registrar_hipotese(codigo,proposicao,estatistica,p_valor,evidencia,limitacao,diretriz,status):
    resultados.append({
        "codigo":codigo,"proposicao":proposicao,"estatistica":estatistica,
        "p_valor":p_valor,"evidencia":evidencia,"limitacao":limitacao,
        "diretriz":diretriz,"status":status,
    })

def spearman_seguro(df,x,y="indicador_t"):
    dd=df[[x,y]].apply(pd.to_numeric,errors="coerce").dropna()
    if len(dd)<3 or dd[x].nunique()<2:return np.nan,np.nan,len(dd)
    r,p=stats.spearmanr(dd[x],dd[y])
    return float(r),float(p),len(dd)

ano_recente=int(mun["ano_alvo"].max())
d=mun[mun["ano_alvo"]==ano_recente].copy()


## 2. H1 · A UF captura heterogeneidade relevante

**Proposição:** diferenças entre UFs representam uma parcela relevante da dispersão municipal.

**Limitação:** a associação combina diferenças de gestão, contexto e instrumento de avaliação.


In [3]:
grupos=[g["indicador_t"].dropna().values for _,g in d.groupby("sigla_uf") if g["indicador_t"].notna().sum()>1]
h,p=stats.kruskal(*grupos)
media=d["indicador_t"].mean(); ss_total=((d["indicador_t"]-media)**2).sum()
ss_entre=sum(len(g)*(g.mean()-media)**2 for g in grupos)
eta2=ss_entre/ss_total
registrar_hipotese(
 "H1","A UF captura heterogeneidade relevante",eta2,p,
 f"Eta² aproximado={eta2:.3f}; Kruskal-Wallis H={h:.1f}",
 "UF não separa efeito de política, contexto e avaliação.",
 "Manter sigla_uf como categórica; agrupar a validação por município.",
 "suportada" if eta2>=.10 and p<.05 else "evidência fraca",
)
print(resultados[-1])


{'codigo': 'H1', 'proposicao': 'A UF captura heterogeneidade relevante', 'estatistica': np.float64(0.3574603980026716), 'p_valor': np.float64(0.0), 'evidencia': 'Eta² aproximado=0.357; Kruskal-Wallis H=1944.3', 'limitacao': 'UF não separa efeito de política, contexto e avaliação.', 'diretriz': 'Manter sigla_uf como categórica; agrupar a validação por município.', 'status': 'suportada'}


## 3. H2 · Histórico recente e IDEB antecipam o resultado


In [4]:
vars_h2=[c for c in [
 "indicador_t1","salto_necessario","ideb_ideb_observado_ultimo",
 "ideb_ideb_nota_portugues_ultimo","ideb_ideb_rendimento_ultimo"
] if c in d.columns]
linhas=[]
for v in vars_h2:
    r,p,n=spearman_seguro(d,v)
    linhas.append({"coluna":v,"rotulo":rotulo(v),"spearman":r,"p_valor":p,"n":n})
tab=pd.DataFrame(linhas).sort_values("spearman",key=lambda s:s.abs(),ascending=False)
display(tab.round(3))
melhor=tab.iloc[0]
registrar_hipotese(
 "H2","Histórico recente e IDEB antecipam o indicador",float(melhor.spearman),float(melhor.p_valor),
 f"Maior |Spearman|: {melhor.rotulo} = {melhor.spearman:.3f}",
 "Correlação não prova causalidade; histórico pode dominar o contexto.",
 "Manter histórico; comparar modelos com e sem histórico.",
 "suportada" if abs(melhor.spearman)>=.3 else "parcial",
)


,coluna,rotulo,spearman,p_valor,n
0,indicador_t1,Alfabetizados no ano anterior (%),0.691,0.0,5323
2,ideb_ideb_observado_ultimo,IDEB mais recente,0.505,0.0,5186
3,ideb_ideb_nota_portugues_ultimo,Nota de Portugues mais recente,0.471,0.0,5188
1,salto_necessario,Avanco necessario para atingir a meta (p.p.),-0.456,0.0,5323
4,ideb_ideb_rendimento_ultimo,Rendimento do IDEB mais recente,0.391,0.0,5258


## 4. H3 · Contexto socioeconômico e estrutura econômica acrescentam sinal


In [5]:
vars_h3=[c for c in [
 "media_inse","pib_per_capita_reais","populacao","densidade_hab_km2",
 "participacao_vab_agro_pct","participacao_vab_industria_pct",
 "participacao_vab_servicos_pct","participacao_vab_adm_publica_pct"
] if c in d.columns]
linhas=[]
for v in vars_h3:
    r,p,n=spearman_seguro(d,v)
    linhas.append({"coluna":v,"rotulo":rotulo(v),"spearman":r,"p_valor":p,"n":n})
tab3=pd.DataFrame(linhas).sort_values("spearman",key=lambda s:s.abs(),ascending=False)
display(tab3.round(3))
max_abs=tab3["spearman"].abs().max()
registrar_hipotese(
 "H3","Contexto socioeconômico e economia acrescentam sinal",float(max_abs),float(tab3.loc[tab3["spearman"].abs().idxmax(),"p_valor"]),
 f"Maior |Spearman| do bloco={max_abs:.3f}",
 "Variáveis municipais são proxies agregadas e podem ocultar desigualdade interna.",
 "Manter o bloco; aplicar log1p a porte e renda; confirmar contribuição por ablação e SHAP.",
 "suportada" if max_abs>=.15 else "evidência fraca",
)


,coluna,rotulo,spearman,p_valor,n
2,populacao,Populacao,-0.279,0.00,5439
4,participacao_vab_agro_pct,Participacao da agropecuaria no PIB (%),0.257,0.00,5439
7,participacao_vab_adm_publica_pct,Participacao da administracao publica no PIB (%),0.215,0.00,5439
5,participacao_vab_industria_pct,Participacao da industria no PIB (%),0.197,0.00,5439
6,participacao_vab_servicos_pct,Participacao dos servicos no PIB (%),0.140,0.00,5439
3,densidade_hab_km2,Densidade demografica (hab./km²),-0.118,0.00,5439
0,media_inse,Nivel socioeconomico medio (INSE),0.093,0.00,5396
1,pib_per_capita_reais,PIB per capita (R$),0.028,0.04,5439


## 5. H4 · Relações não lineares justificam modelos de árvore


In [6]:
# Compara ajuste linear e monotonia por quantis para variáveis disponíveis.
vars_h4=[c for c in ["salto_necessario","populacao","media_inse","ideb_ideb_observado_ultimo"] if c in d.columns]
linhas=[]
for v in vars_h4:
    dd=d[[v,"indicador_t"]].apply(pd.to_numeric,errors="coerce").dropna()
    if len(dd)<20: continue
    dd["faixa"]=pd.qcut(dd[v],8,duplicates="drop")
    med=dd.groupby("faixa",observed=True)["indicador_t"].median()
    diffs=np.diff(med.values)
    mudancas=int(np.sum(np.sign(diffs[1:])!=np.sign(diffs[:-1]))) if len(diffs)>1 else 0
    linhas.append({"variavel":rotulo(v),"mudancas_de_direcao":mudancas,"amplitude_mediana":med.max()-med.min()})
tab4=pd.DataFrame(linhas)
display(tab4)
registrar_hipotese(
 "H4","Algumas relações são não lineares",float(tab4["mudancas_de_direcao"].max() if not tab4.empty else np.nan),np.nan,
 "Curvas por octis apresentam saturação ou mudança de inclinação.",
 "Quantis são descritivos e sensíveis à composição territorial.",
 "Comparar regressão linear regularizada com Random Forest; usar backtest temporal.",
 "a testar na modelagem",
)


,variavel,mudancas_de_direcao,amplitude_mediana
0,Avanco necessario para atingir a meta (p.p.),1,25.0
1,Populacao,0,17.0
2,Nivel socioeconomico medio (INSE),3,13.0
3,IDEB mais recente,2,32.0


## 6. H5 · Features com cobertura temporal insuficiente devem ficar fora do modelo principal


In [7]:
restritas=[c for c in ["indicador_t2","variacao_t1","meta_t1","gap_meta_t1"] if c in mun.columns]
cobertura=[]
for v in restritas:
    for ano,g in mun.groupby("ano_alvo"):
        cobertura.append({"variavel":v,"ano":int(ano),"disponivel_pct":100*g[v].notna().mean()})
tab5=pd.DataFrame(cobertura)
display(tab5.pivot(index="variavel",columns="ano",values="disponivel_pct").round(1))
min_cob=tab5.groupby("variavel")["disponivel_pct"].min().min() if not tab5.empty else 100
registrar_hipotese(
 "H5","Features de dois anos têm cobertura temporal insuficiente",float(min_cob),np.nan,
 f"Menor cobertura ano-feature={min_cob:.1f}%",
 "A série histórica ainda é curta.",
 "Excluir do modelo principal até existirem valores no ano de treino; manter na ABT.",
 "suportada" if min_cob<50 else "não suportada",
)


ano,2024,2025
variavel,,
gap_meta_t1,0.0,97.9
indicador_t2,0.0,97.9
meta_t1,0.0,100.0
variacao_t1,0.0,95.8


## 7. H6 · O modelo de aluno tem pouco sinal individual disponível


In [8]:
def amostrar_alunos(caminho: Path, n: int = 100_000, seed: int = SEED) -> pd.DataFrame:
    """Amostra aproximada por ano e target sem carregar o Parquet inteiro."""
    pf = pq.ParquetFile(caminho)
    total = pf.metadata.num_rows
    frac = min(1.0, n / total)
    rng = np.random.default_rng(seed)
    partes = []
    for batch in pf.iter_batches(batch_size=50_000, use_threads=False):
        lote = batch.to_pandas()
        if "alfabetizado" in lote:
            lote = lote.dropna(subset=["alfabetizado"])
        qtd = min(len(lote), max(1, int(np.ceil(len(lote) * frac * 1.2))))
        if qtd:
            idx = rng.choice(len(lote), size=qtd, replace=False)
            partes.append(lote.iloc[idx])
    amostra = pd.concat(partes, ignore_index=True)
    if len(amostra) > n:
        # novo balanceamento por ano e alvo
        estratos=[]
        for _, g in amostra.groupby(["ano", "alfabetizado"], observed=True):
            q=max(1, round(n*len(g)/len(amostra)))
            estratos.append(g.sample(n=min(q,len(g)), random_state=seed))
        amostra=pd.concat(estratos,ignore_index=True)
        if len(amostra)>n:
            amostra=amostra.sample(n=n,random_state=seed)
    return amostra.sample(frac=1,random_state=seed).reset_index(drop=True)

amostra_aluno = amostrar_alunos(ARQ_ALU, n=100_000)
print("Amostra de alunos:", amostra_aluno.shape)


Amostra de alunos: (99999, 61)


In [9]:
a=amostra_aluno.dropna(subset=["id_municipio","id_escola","alfabetizado"]).copy()
ano=max(a["ano"]); a=a[a["ano"]==ano]
media_mun=a.groupby("id_municipio")["alfabetizado"].transform("mean")
media_esc=a.groupby("id_escola")["alfabetizado"].transform("mean")
decomp=pd.Series({
 "entre municípios":media_mun.var(),
 "entre escolas no município":(media_esc-media_mun).var(),
 "entre alunos na escola":(a["alfabetizado"]-media_esc).var(),
})
percentual=100*decomp/decomp.sum()
display(percentual.round(1).to_frame("variancia_pct_aproximada"))
parte_individual=float(percentual["entre alunos na escola"])
registrar_hipotese(
 "H6","A maior parte da variação está entre alunos da mesma escola",parte_individual,np.nan,
 f"Parcela individual aproximada na amostra={parte_individual:.1f}%",
 "Decomposição em amostra e sem covariáveis individuais; não é um modelo multinível formal.",
 "Interpretar o modelo como priorização territorial/escolar; usar AUC e recall de não alfabetizados.",
 "suportada" if parte_individual>=50 else "não suportada",
)


,variancia_pct_aproximada
entre municípios,18.0
entre escolas no município,47.1
entre alunos na escola,34.9


## 8. Diretrizes consolidadas

A tabela abaixo substitui afirmações fixas por resultados calculados. As diretrizes devem ser confrontadas com o backtest dos modelos.


In [10]:
decisoes=pd.DataFrame(resultados)
display(decisoes)

# Regras adicionais de segurança contra leakage.
regras=pd.DataFrame([
 {"tema":"target municipal","diretriz":"indicador_t, atingiu_meta_t e gap_meta_t não entram em X"},
 {"tema":"target aluno","diretriz":"proficiencia não entra em X para prever alfabetizado"},
 {"tema":"validação","diretriz":"separação temporal e grupos de município"},
 {"tema":"ausência","diretriz":"imputação dentro do pipeline e remoção de colunas totalmente nulas no treino"},
 {"tema":"saída","diretriz":"probabilidade e ranking, além da classe"},
])
display(regras)


,codigo,proposicao,estatistica,p_valor,evidencia,limitacao,diretriz,status
0,H1,A UF captura heterogeneidade relevante,0.357460,0.000000e+00,Eta² aproximado=0.357; Kruskal-Wallis H=1944.3,"UF não separa efeito de política, contexto e a...",Manter sigla_uf como categórica; agrupar a val...,suportada
1,H2,Histórico recente e IDEB antecipam o indicador,0.691273,0.000000e+00,Maior |Spearman|: Alfabetizados no ano anterio...,Correlação não prova causalidade; histórico po...,Manter histórico; comparar modelos com e sem h...,suportada
2,H3,Contexto socioeconômico e economia acrescentam...,0.278827,1.087352e-97,Maior |Spearman| do bloco=0.279,Variáveis municipais são proxies agregadas e p...,Manter o bloco; aplicar log1p a porte e renda;...,suportada
3,H4,Algumas relações são não lineares,3.000000,NaN,Curvas por octis apresentam saturação ou mudan...,Quantis são descritivos e sensíveis à composiç...,Comparar regressão linear regularizada com Ran...,a testar na modelagem
4,H5,Features de dois anos têm cobertura temporal i...,0.000000,NaN,Menor cobertura ano-feature=0.0%,A série histórica ainda é curta.,Excluir do modelo principal até existirem valo...,suportada
5,H6,A maior parte da variação está entre alunos da...,34.878593,NaN,Parcela individual aproximada na amostra=34.9%,Decomposição em amostra e sem covariáveis indi...,Interpretar o modelo como priorização territor...,não suportada


,tema,diretriz
0,target municipal,"indicador_t, atingiu_meta_t e gap_meta_t não e..."
1,target aluno,proficiencia não entra em X para prever alfabe...
2,validação,separação temporal e grupos de município
3,ausência,imputação dentro do pipeline e remoção de colu...
4,saída,"probabilidade e ranking, além da classe"


## 9. Encaminhamento

Os notebooks de modelagem devem comparar, no mínimo:

- baseline linear;
- modelo de árvore;
- versão com e sem histórico;
- desempenho em validação agrupada e no backtest temporal;
- estabilidade por UF, rede e ano;
- importância por permutação e SHAP após a escolha do modelo.
